<a href="https://colab.research.google.com/github/ErickJester/expo-escom/blob/main/00_conteo_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📊 Conteo del Dataset — ExpoEscom
Cuenta imágenes dentro de una carpeta de Drive (incluye "Compartido conmigo").

**Flujo:**
1. Lee todo lo que hay dentro del `FOLDER_ID` pegado.
2. Arma un índice numerado de las subcarpetas encontradas.
3. Te pregunta cuál carpeta quieres contabilizar.

Solo lectura: no descarga ni modifica nada.

In [ ]:
VERSION = '2.0.0'

print('═' * 50)
print('📊 Conteo Dataset — ExpoEscom')
print(f'v{VERSION}')
print('═' * 50)

from google.colab import auth
from googleapiclient.discovery import build

auth.authenticate_user()
service = build('drive', 'v3')
print('✅ Autenticado con Drive API')

In [ ]:
# ════ ÚNICA CONFIGURACIÓN NECESARIA ══════════════════════════

FOLDER_ID = '1YwNZMW67NYGMb_g-74kG5gb2u_m_3c1J'

EXTENSIONES_IMAGEN = {'.jpg', '.jpeg', '.png', '.webp', '.bmp', '.gif'}
print(f'Folder ID : {FOLDER_ID}')

In [ ]:
# ════ HELPERS DE DRIVE API ════════════════════════════════════
from pathlib import Path

_ARGS = dict(supportsAllDrives=True, includeItemsFromAllDrives=True,
             pageSize=1000)
_FOLDER_MIME = 'application/vnd.google-apps.folder'


def listar_contenido(parent_id):
    """Lee el nivel directo de parent_id.
    Devuelve (subcarpetas, n_imagenes_sueltas):
      subcarpetas = lista de dicts {id, name} ordenada por nombre.
      n_imagenes_sueltas = imágenes que cuelgan directo de la raíz."""
    subcarpetas, sueltas, token = [], 0, None
    while True:
        resp = service.files().list(
            q=f"'{parent_id}' in parents and trashed=false",
            fields='nextPageToken, files(id,name,mimeType)',
            pageToken=token, **_ARGS).execute()
        for f in resp.get('files', []):
            if f['mimeType'] == _FOLDER_MIME:
                subcarpetas.append({'id': f['id'], 'name': f['name']})
            elif (f['mimeType'].startswith('image/') or
                  Path(f['name']).suffix.lower() in EXTENSIONES_IMAGEN):
                sueltas += 1
        token = resp.get('nextPageToken')
        if not token:
            break
    subcarpetas.sort(key=lambda c: c['name'].lower())
    return subcarpetas, sueltas


def contar_imagenes(folder_id):
    """Cuenta recursivamente los archivos-imagen bajo folder_id."""
    total, stack = 0, [folder_id]
    while stack:
        fid, token = stack.pop(), None
        while True:
            resp = service.files().list(
                q=f"'{fid}' in parents and trashed=false",
                fields='nextPageToken, files(id,name,mimeType)',
                pageToken=token, **_ARGS).execute()
            for f in resp.get('files', []):
                if f['mimeType'] == _FOLDER_MIME:
                    stack.append(f['id'])
                elif (f['mimeType'].startswith('image/') or
                      Path(f['name']).suffix.lower() in EXTENSIONES_IMAGEN):
                    total += 1
            token = resp.get('nextPageToken')
            if not token:
                break
    return total


print('✅ Helpers listos')

## Pasos 1 y 2 — Leer la carpeta y construir el índice

In [ ]:
# ── Leer todo el contenido del FOLDER_ID y armar el índice ────
raiz = service.files().get(
    fileId=FOLDER_ID, fields='name', supportsAllDrives=True).execute()

print(f'📂 Carpeta raíz : {raiz["name"]}')
print('Leyendo contenido…\n')

_subcarpetas, _sueltas = listar_contenido(FOLDER_ID)

# Índice: 0 = toda la raíz, 1..N = cada subcarpeta
INDICE = {0: {'name': f'(TODA la carpeta «{raiz["name"]}»)', 'id': FOLDER_ID}}
for i, c in enumerate(_subcarpetas, 1):
    INDICE[i] = c

print('═' * 50)
print('ÍNDICE DE CARPETAS')
print('═' * 50)
for num, c in INDICE.items():
    print(f'  [{num:>2}]  {c["name"]}')
print('═' * 50)
print(f'Subcarpetas : {len(_subcarpetas)}')
if _sueltas:
    print(f'⚠️  Además hay {_sueltas:,} imágenes sueltas en la raíz '
          f'(se cuentan con la opción [0]).')

## Paso 3 — Elegir qué carpeta contabilizar

In [ ]:
# ── Preguntar al usuario qué carpeta contar ───────────────────
print('Carpetas disponibles:')
for num, c in INDICE.items():
    print(f'  [{num:>2}]  {c["name"]}')

while True:
    sel = input('\n¿Qué carpeta quieres contabilizar? (número): ').strip()
    if sel.isdigit() and int(sel) in INDICE:
        sel = int(sel)
        break
    print('   ⚠️  Número inválido, intenta de nuevo.')

elegida = INDICE[sel]
print(f'\n⏳ Contando imágenes en: {elegida["name"]}…')

n = contar_imagenes(elegida['id'])

print('\n' + '═' * 50)
print(f'  Carpeta : {elegida["name"]}')
print(f'  Imágenes: {n:,}')
print('═' * 50)